# Demo: KI-Notebook-Workspace im Container

Dieses Notebook demonstriert, warum Container für KI-nahe Arbeitsumgebungen nützlich sind.

Der Container enthält bereits Python, JupyterLab, NumPy, pandas, matplotlib, scikit-learn, pandoc und eine LaTeX-Distribution. Damit kann die Umgebung auf verschiedenen Rechnern gleich gestartet werden, ohne lokal alle Werkzeuge einzeln zu installieren.

## 1. Umgebung prüfen

Zuerst prüfen wir, welche Python-Version im Container läuft.

In [ ]:
import sys
print(sys.version)

## 2. Kleine Beispieldaten erzeugen

Wir erzeugen einen einfachen Datensatz mit zwei Merkmalen und einer Zielgröße. Das ist kein komplexes KI-Beispiel, sondern dient dazu, die vorinstallierten Bibliotheken zu nutzen.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 80
lernzeit = rng.normal(loc=6, scale=2, size=n).clip(0, 12)
vorwissen = rng.normal(loc=50, scale=15, size=n).clip(0, 100)
punkte = 20 + 4.2 * lernzeit + 0.45 * vorwissen + rng.normal(0, 8, size=n)
punkte = punkte.clip(0, 100)

df = pd.DataFrame({
    "lernzeit_stunden": lernzeit.round(1),
    "vorwissen_index": vorwissen.round(1),
    "punkte": punkte.round(1),
})

df.head()

## 3. Daten visualisieren

Matplotlib ist bereits im Container installiert. Auf dem Host-System muss dafür nichts eingerichtet werden.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.scatter(df["lernzeit_stunden"], df["punkte"])
plt.xlabel("Lernzeit in Stunden")
plt.ylabel("Punkte")
plt.title("Beispieldaten")
plt.grid(True)
plt.show()

## 4. Einfaches ML-Modell trainieren

Nun trainieren wir eine lineare Regression mit scikit-learn.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

X = df[["lernzeit_stunden", "vorwissen_index"]]
y = df["punkte"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("R²:", round(r2_score(y_test, pred), 2))
print("Koeffizienten:", dict(zip(X.columns, model.coef_.round(2))))

## 5. Datei im Workspace speichern

Wenn der Container mit einem Volume gestartet wurde, bleiben Dateien im gemounteten Verzeichnis erhalten.

Beispielstart auf macOS/Linux:

```bash
docker run --rm -p 8888:8888 -v "$PWD/workspace":/workspace ai-workspace
```

Die folgende Zelle speichert die Beispieldaten als CSV-Datei.

In [ ]:
output_path = "/workspace/beispieldaten.csv"
df.to_csv(output_path, index=False)
print(f"Gespeichert unter: {output_path}")

## 6. Systemwerkzeug im Container verwenden

Das Image enthält zusätzlich `pandoc`. Solche Programme müssten ohne Container separat auf dem Rechner installiert werden.

In [ ]:
!pandoc --version | head -n 2

## 7. Markdown-Datei mit Pandoc erzeugen

Die nächste Zelle schreibt eine kleine Markdown-Datei und wandelt sie mit `pandoc` in HTML um.

In [ ]:
from pathlib import Path

md_path = Path("/workspace/bericht.md")
html_path = Path("/workspace/bericht.html")

md_path.write_text(
    "# Beispielbericht\n\n"
    "Dieser Bericht wurde in einem JupyterLab-Container erzeugt.\n\n"
    f"Der Datensatz enthält **{len(df)}** Zeilen.\n",
    encoding="utf-8",
)

!pandoc /workspace/bericht.md -o /workspace/bericht.html

print(f"Markdown: {md_path}")
print(f"HTML:     {html_path}")

## 8. Optional: Notebook als PDF exportieren

Wenn LaTeX vollständig genug installiert ist, kann Jupyter das Notebook als PDF exportieren:

```bash
jupyter nbconvert --to pdf notebooks/demo.ipynb
```

Falls dabei ein LaTeX-Paket fehlt, muss es im Dockerfile ergänzt werden. Genau das ist ein typischer Grund, warum Container nützlich sind: Die benötigten Systempakete werden explizit dokumentiert.